In [0]:
# ===========================================
# Notebook Name:
# 02_create_v2_tables
#
# Purpose:
# Single source of truth for all v2 table DDL.
# CREATE TABLE IF NOT EXISTS only: tables that
# already match the v2 shape are left untouched.
#
# Scope:
# pokemon.bronze / silver / gold / ops
# (17 tables total, see docs/migration_plan #12)
# ===========================================

In [0]:
CATALOG_NAME = "pokemon"

print("Catalog:", CATALOG_NAME)

In [ ]:
# ---- Bronze ----

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG_NAME}.bronze.tournament_list_raw
(
    run_id STRING NOT NULL,
    page_number INT NOT NULL,
    offset INT NOT NULL,
    page_size INT NOT NULL,
    request_url STRING NOT NULL,
    request_params STRING NOT NULL,
    response_status INT NOT NULL,
    api_code INT,
    payload STRING NOT NULL,
    response_hash STRING NOT NULL,
    event_count INT NOT NULL,
    eligible_event_count INT NOT NULL,
    page_min_event_date DATE,
    page_max_event_date DATE,
    contains_eligible_events BOOLEAN NOT NULL,
    all_events_before_cutoff BOOLEAN NOT NULL,
    elapsed_ms BIGINT,
    fetched_at TIMESTAMP NOT NULL,
    ingestion_date DATE NOT NULL
)
USING DELTA
COMMENT 'Raw per-page tournament list API responses'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG_NAME}.bronze.event_result_raw
(
    tournament_id STRING NOT NULL,
    source_url STRING NOT NULL,
    http_status INT,
    payload STRING NOT NULL,
    payload_format STRING NOT NULL,
    response_hash STRING NOT NULL,
    scraped_at TIMESTAMP NOT NULL,
    ingestion_date DATE NOT NULL
)
USING DELTA
COMMENT 'Raw tournament result API responses, one row per tournament_id and response_hash'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG_NAME}.bronze.deck_raw
(
    deck_code STRING NOT NULL,
    source_url STRING NOT NULL,
    http_status INT,
    payload STRING NOT NULL,
    payload_format STRING NOT NULL,
    response_hash STRING NOT NULL,
    scraped_at TIMESTAMP NOT NULL,
    ingestion_date DATE NOT NULL
)
USING DELTA
COMMENT 'Raw deck HTML responses, one row per deck_code and response_hash'
""")

# NOTE: request_id identifies a single fetch attempt
# (one per tournament/deck/list-crawl request);
# pipeline_run_id identifies the Workflow run that
# request belongs to, shared across every scrape_log
# row written by every ingest notebook/task in that
# run. See notebooks/05_ops/00_initialize_pipeline_run
# for where pipeline_run_id is generated.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG_NAME}.bronze.scrape_log
(
    request_id STRING NOT NULL,
    pipeline_run_id STRING NOT NULL,
    source_type STRING NOT NULL,
    source_id STRING NOT NULL,
    request_url STRING,
    http_status INT,
    status STRING NOT NULL,
    elapsed_ms BIGINT,
    records_found INT,
    error_type STRING,
    error_message STRING,
    scraped_at TIMESTAMP NOT NULL,
    ingestion_date DATE NOT NULL
)
USING DELTA
COMMENT 'Unified fetch-attempt log across tournament_list / event_result / deck sources'
""")

print("Bronze tables ensured.")

In [ ]:
# ---- Silver ----
# NOTE: silver.tournaments legacy columns
# (event_title, event_type_id, event_type_title,
# event_date_display, prefecture_name, venue,
# address, shop_name, league, regulation,
# capacity, result_count, source_url,
# response_hash) are removed by
# 00_migration/05_drop_legacy_tournament_columns
# after its backfill-completeness check passes.
# This DDL reflects the v2-only shape so
# CREATE TABLE IF NOT EXISTS matches production
# for any new environment.

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG_NAME}.silver.tournaments
(
    tournament_id STRING NOT NULL,
    shop_id STRING,
    source_scraped_at TIMESTAMP,
    updated_at TIMESTAMP,
    event_date_text STRING,
    event_date_week STRING,
    event_started_at STRING,
    event_ended_at STRING,
    venue_name STRING,
    prefecture STRING,
    date_id INT,
    event_type STRING,
    event_json STRING,
    event_hash STRING,
    source_run_id STRING,
    source_offset INT,
    source_response_hash STRING,
    source_fetched_at TIMESTAMP,
    first_seen_at TIMESTAMP,
    last_changed_at TIMESTAMP,
    event_date DATE
)
USING DELTA
COMMENT 'v2 canonical columns: venue_name/prefecture/date_id/event_type/event_hash/event_date. Legacy columns removed by 05_drop_legacy_tournament_columns.'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG_NAME}.silver.tournament_results
(
    tournament_id STRING NOT NULL,
    rank INT NOT NULL,
    player_id STRING,
    player_name STRING,
    area STRING,
    point INT,
    show_profile BOOLEAN,
    deck_code STRING,
    source_url STRING,
    response_hash STRING,
    source_scraped_at TIMESTAMP,
    updated_at TIMESTAMP,
    deck_url STRING,
    deck_print_url STRING,
    deck_hash STRING,
    deck_hash_version STRING
)
USING DELTA
COMMENT 'One row per tournament_id and rank'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG_NAME}.silver.decks
(
    deck_hash STRING NOT NULL,
    deck_hash_version STRING NOT NULL,
    deck_code STRING NOT NULL,
    source_url STRING,
    response_hash STRING,
    source_scraped_at TIMESTAMP,
    total_cards INT,
    card_type_rows BIGINT,
    unique_card_names BIGINT,
    is_valid BOOLEAN,
    canonical_string STRING,
    updated_at TIMESTAMP
)
USING DELTA
COMMENT 'One row per deck_code, deck_hash is the functional identity (ADR-001)'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG_NAME}.silver.deck_cards
(
    deck_code STRING NOT NULL,
    deck_hash STRING NOT NULL,
    category STRING NOT NULL,
    card_name STRING NOT NULL,
    quantity INT NOT NULL,
    expansion STRING,
    collection_number STRING,
    source_url STRING,
    response_hash STRING,
    source_scraped_at TIMESTAMP,
    updated_at TIMESTAMP
)
USING DELTA
COMMENT 'One row per deck_code x category x card_name'
""")

print("Silver tables ensured.")

In [ ]:
# ---- Gold ----

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG_NAME}.gold.card_usage
(
    card_name STRING NOT NULL,
    category STRING NOT NULL,
    decks_using_card BIGINT,
    total_copies BIGINT,
    avg_copies_when_used DOUBLE,
    min_copies_when_used INT,
    max_copies_when_used INT,
    total_decks INT,
    inclusion_rate DOUBLE,
    inclusion_rate_pct DOUBLE,
    updated_at TIMESTAMP
)
USING DELTA
COMMENT 'Card usage metrics excluding Basic Energy cards'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG_NAME}.gold.deck_registry
(
    deck_hash STRING NOT NULL,
    deck_hash_version STRING NOT NULL,
    representative_deck_code STRING,
    deck_codes ARRAY<STRING>,
    total_cards INT,
    unique_card_names BIGINT,
    card_type_rows BIGINT,
    first_scraped_at TIMESTAMP,
    last_scraped_at TIMESTAMP,
    canonical_string STRING,
    is_valid BOOLEAN,
    first_seen_date DATE,
    last_seen_date DATE,
    tournament_count BIGINT,
    result_appearance_count BIGINT,
    official_deck_code_count BIGINT,
    unique_player_count BIGINT,
    best_rank INT,
    average_rank DOUBLE,
    median_rank INT,
    win_count BIGINT,
    top4_count BIGINT,
    top8_count BIGINT,
    win_rate DOUBLE,
    top4_rate DOUBLE,
    top8_rate DOUBLE,
    win_rate_pct DOUBLE,
    top4_rate_pct DOUBLE,
    top8_rate_pct DOUBLE,
    updated_at TIMESTAMP
)
USING DELTA
COMMENT 'One row per unique functional deck (deck_hash)'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG_NAME}.gold.deck_pokemon_features
(
    deck_hash STRING NOT NULL,
    deck_hash_version STRING NOT NULL,
    representative_deck_code STRING,
    card_name STRING NOT NULL,
    quantity INT NOT NULL,
    is_present INT NOT NULL,
    source_deck_code_count BIGINT,
    source_deck_codes ARRAY<STRING>,
    expansions ARRAY<STRING>,
    collection_numbers ARRAY<STRING>,
    updated_at TIMESTAMP
)
USING DELTA
COMMENT 'One row per deck_hash x Pokemon card name'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG_NAME}.gold.deck_similarity
(
    deck_hash_a STRING NOT NULL,
    deck_code_a STRING,
    best_rank_a INT,
    average_rank_a DOUBLE,
    deck_hash_b STRING NOT NULL,
    deck_code_b STRING,
    best_rank_b INT,
    average_rank_b DOUBLE,
    weighted_intersection BIGINT,
    weighted_union BIGINT,
    shared_pokemon_names BIGINT,
    union_pokemon_names BIGINT,
    total_quantity_difference BIGINT,
    weighted_jaccard_similarity DOUBLE,
    weighted_jaccard_pct DOUBLE,
    presence_jaccard_similarity DOUBLE,
    presence_jaccard_pct DOUBLE,
    updated_at TIMESTAMP
)
USING DELTA
COMMENT 'One row per unique deck_hash pair (deck_hash_a < deck_hash_b)'
""")

# NOTE: deck_archetypes is a machine-generated
# cluster assignment table only. It intentionally
# has no archetype_name column: that column was
# removed because the notebook that builds this
# table rebuilds it from scratch on every run, which
# silently erased any human-entered name. Human
# identity now lives in gold.archetype_catalog,
# joined via gold.archetype_cluster_mapping (keyed
# by the stable cluster_signature, not the unstable
# cluster_id). See gold.v_deck_archetypes_named.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG_NAME}.gold.deck_archetypes
(
    deck_hash STRING NOT NULL,
    deck_hash_version STRING NOT NULL,
    representative_deck_code STRING,
    cluster_id INT NOT NULL,
    cluster_signature STRING NOT NULL,
    cluster_size INT NOT NULL,
    model_name STRING NOT NULL,
    model_run_id STRING NOT NULL,
    cluster_count INT NOT NULL,
    silhouette_score DOUBLE,
    clustered_at TIMESTAMP NOT NULL,
    updated_at TIMESTAMP
)
USING DELTA
COMMENT 'Rebuilt atomically every run (CREATE OR REPLACE TABLE AS SELECT): one row per deck_hash for the latest clustering. No archetype_name column by design; see gold.v_deck_archetypes_named.'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG_NAME}.gold.archetype_catalog
(
    archetype_id STRING NOT NULL,
    archetype_name STRING NOT NULL,
    display_name STRING,
    description STRING,
    status STRING,
    reviewed_by STRING,
    reviewed_at TIMESTAMP,
    created_at TIMESTAMP
)
USING DELTA
COMMENT 'Human-curated archetype identities. Edited only via 04_ml/02_review_archetype_mapping, never by ML rebuild jobs.'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG_NAME}.gold.archetype_cluster_mapping
(
    model_run_id STRING NOT NULL,
    cluster_signature STRING NOT NULL,
    cluster_id INT NOT NULL,
    archetype_id STRING,
    mapping_status STRING NOT NULL,
    confidence DOUBLE,
    reviewed_at TIMESTAMP
)
USING DELTA
COMMENT 'One row per model_run_id x cluster_signature. Links an unstable per-run cluster_id to a stable archetype_id once reviewed.'
""")

spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG_NAME}.gold.v_deck_archetypes_named AS
SELECT
    da.*,
    acm.archetype_id,
    ac.archetype_name,
    ac.display_name,
    acm.mapping_status,
    acm.confidence AS mapping_confidence
FROM {CATALOG_NAME}.gold.deck_archetypes AS da
LEFT JOIN {CATALOG_NAME}.gold.archetype_cluster_mapping AS acm
    ON da.model_run_id = acm.model_run_id
    AND da.cluster_signature = acm.cluster_signature
LEFT JOIN {CATALOG_NAME}.gold.archetype_catalog AS ac
    ON acm.archetype_id = ac.archetype_id
""")

print("Gold tables ensured.")

In [0]:
# ---- Ops ----

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG_NAME}.ops.result_fetch_targets
(
    tournament_id STRING,
    event_date DATE,
    event_title STRING,
    fetch_reason STRING,
    priority INT,
    has_reported_results BOOLEAN,
    has_raw_result BOOLEAN,
    has_silver_results BOOLEAN,
    result_fetched_at TIMESTAMP,
    tournament_changed_at TIMESTAMP,
    latest_fetch_status STRING,
    latest_fetch_error STRING,
    latest_result_response_hash STRING,
    empty_retry_available_at TIMESTAMP,
    identified_at TIMESTAMP
)
USING DELTA
COMMENT 'Overwritten every run: tournaments whose result API should be (re)fetched'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG_NAME}.ops.deck_fetch_targets
(
    deck_code STRING NOT NULL,
    tournament_id STRING,
    fetch_reason STRING,
    priority INT,
    has_raw_deck BOOLEAN,
    has_silver_deck BOOLEAN,
    deck_fetched_at TIMESTAMP,
    deck_changed_at TIMESTAMP,
    latest_fetch_status STRING,
    latest_fetch_error STRING,
    latest_deck_response_hash STRING,
    identified_at TIMESTAMP
)
USING DELTA
COMMENT 'Overwritten every run: deck_codes that should be (re)fetched. Priority: manual_refetch > retry_error > parse_error > never_fetched'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG_NAME}.ops.pipeline_run_log
(
    pipeline_run_id STRING NOT NULL,
    job_run_id STRING,
    task_name STRING NOT NULL,
    status STRING NOT NULL,
    started_at TIMESTAMP,
    finished_at TIMESTAMP,
    elapsed_ms BIGINT,
    input_count BIGINT,
    insert_count BIGINT,
    update_count BIGINT,
    skip_count BIGINT,
    error_count BIGINT,
    error_message STRING
)
USING DELTA
COMMENT 'One row per pipeline_run_id x task_name (grain per docs/operation_workflow.md #9)'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG_NAME}.ops.pipeline_state
(
    pipeline_name STRING NOT NULL,
    stage_name STRING NOT NULL,
    last_run_at TIMESTAMP,
    last_run_status STRING,
    last_success_at TIMESTAMP,
    watermark_value STRING,
    metrics_json STRING,
    consecutive_failure_count INT
)
USING DELTA
COMMENT 'One row per pipeline_name x stage_name, MERGE target for watermarks and last-run metrics'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG_NAME}.ops.migration_inventory
(
    catalog STRING NOT NULL,
    schema STRING NOT NULL,
    table_name STRING NOT NULL,
    column_name STRING,
    data_type STRING,
    nullable BOOLEAN,
    row_count BIGINT,
    created_at TIMESTAMP,
    last_modified TIMESTAMP,
    v2_action STRING NOT NULL,
    inventoried_at TIMESTAMP NOT NULL
)
USING DELTA
COMMENT 'Snapshot of current-state table inventory vs. v2 migration_plan target shape. Overwritten on every run.'
""")

print("Ops tables ensured.")

In [0]:
display(
    spark.sql(
        f"SHOW TABLES IN {CATALOG_NAME}.bronze"
    )
)

In [0]:
display(
    spark.sql(
        f"SHOW TABLES IN {CATALOG_NAME}.ops"
    )
)